In [ ]:
!pip install -q ultralytics opencv-python-headless pandas pyyaml


In [ ]:
from pathlib import Path
import os

PROJETO = Path('/content/ONIBUS_CHUVA_SP')
os.chdir(PROJETO)
MODELO = PROJETO / 'models' / 'best.pt'
DATASET = PROJETO / 'dataset' / 'data.yaml'
VIDEO_ENTRADA = PROJETO / 'amostras' / 'video_ponto_onibus.mp4'
SAIDAS = PROJETO / 'saidas'
SAIDAS.mkdir(parents=True, exist_ok=True)
VIDEO_SAIDA = SAIDAS / 'avaliacao_demo.mp4'
CSV_SAIDA = SAIDAS / 'avaliacao_contagens.csv'

for caminho in [MODELO, DATASET, VIDEO_ENTRADA]:
    if not caminho.exists():
        raise FileNotFoundError(caminho)

print(PROJETO)
print(MODELO)
print(DATASET)
print(VIDEO_ENTRADA)


In [ ]:
from ultralytics import YOLO

modelo = YOLO(str(MODELO))
print(modelo.names)


In [ ]:
metricas = modelo.val(
    data=str(DATASET),
    split='test',
    imgsz=640,
    plots=True,
    project=str(SAIDAS),
    name='avaliacao_metricas',
    exist_ok=True,
)

print(f'Precisão: {metricas.box.mp * 100:.2f}%')
print(f'Recall: {metricas.box.mr * 100:.2f}%')
print(f'mAP50: {metricas.box.map50 * 100:.2f}%')
print(f'mAP50-95: {metricas.box.map * 100:.2f}%')


In [ ]:
import subprocess
import sys

comando = [
    sys.executable,
    str(PROJETO / 'src' / 'detectar_tempo_real.py'),
    '--pesos', str(MODELO),
    '--fonte', str(VIDEO_ENTRADA),
    '--anonimizar',
    '--sem-janela',
    '--salvar-video', str(VIDEO_SAIDA),
    '--csv', str(CSV_SAIDA),
    '--conf', '0.35',
    '--imgsz', '640',
    '--intervalo', '1',
]

execucao = subprocess.run(comando, cwd=str(PROJETO), text=True, capture_output=True)
print(execucao.stdout)
if execucao.returncode != 0:
    print(execucao.stderr)
    raise RuntimeError('Falha na execução')


In [ ]:
from IPython.display import Video, display

display(Video(str(VIDEO_SAIDA), embed=True, width=900))


In [ ]:
import pandas as pd

dados = pd.read_csv(CSV_SAIDA)
display(dados.head(10))
print(f'Linhas: {len(dados)}')
print(CSV_SAIDA)
